# CV Processing Pipeline — Qwen 3 4B

**External datasets required (add via Kaggle UI):**
- `cv_extractor_v2.py` — rule-based segmentation extractor
- `cv_parser_output.jsonl` — your CV parser output dataset

**Pipeline:**
1. **Extract** — `cv_extractor_v2.py` segments raw CV text into candidate blocks
2. **Deduplicate** — drops candidates already captured by the parser
3. **Structure** — Qwen 3 4B (local, 4-bit quantized) converts remaining candidates into a clean schema, validated against an explicit output contract
4. **Checkpointed run** — re-running the notebook resumes from the last completed CV instead of reprocessing or duplicating rows

## 1. Install dependencies

In [1]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 60.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.2

## 2. Imports & paths

In [2]:
import sys
import json
import re
import time
import dataclasses
import difflib
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ── adjust these if your dataset slugs differ ──────────────────────────────
EXTRACTOR_DIR   = Path("/kaggle/input/datasets/xmohamedsaber/extractor")          # contains cv_extractor_v2.py
DATA_JSONL      = Path("/kaggle/input/datasets/xmohamedsaber/cvs-data/cv_parser_output.jsonl")  # fixed: was "//kaggle/..."
OUTPUT_JSONL    = Path("/kaggle/working/final_candidates_qwen3.jsonl")

# ── pipeline config ────────────────────────────────────────────────────────
START_INDEX = 0
NUM_CVS     = 50

# ── add extractor to path ──────────────────────────────────────────────────
sys.path.insert(0, str(EXTRACTOR_DIR))
from cv_extractor_v2 import extract_candidates
print("cv_extractor_v2 loaded ✓")

cv_extractor_v2 loaded ✓


## 3. Load Qwen 3 4B (4-bit quantized)

In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ------------------------------------------------------------------
# Login to Hugging Face using Kaggle Secret
# ------------------------------------------------------------------
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

# ------------------------------------------------------------------
# Model configuration
# ------------------------------------------------------------------
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Adjust per GPU type:
# T4 = 15 GiB  →  14 GiB budget (1 GiB headroom)
# P100/V100 = 16 GiB  →  14 GiB budget (2 GiB headroom)
MAX_MEMORY = {
    0: "13GiB",   # slightly conservative to avoid OOM during generation
    1: "13GiB",
    "cpu": "24GiB",
}

# ------------------------------------------------------------------
# Load tokenizer
# ------------------------------------------------------------------
print(f"Loading tokenizer for {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=hf_token,
    trust_remote_code=True,
)

# ------------------------------------------------------------------
# Load model across 2 GPUs
# ------------------------------------------------------------------
print(f"Loading {MODEL_ID} across 2 GPUs...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=hf_token,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory=MAX_MEMORY,
    trust_remote_code=True,
    # torch_dtype removed — bnb_4bit_compute_dtype handles precision
)
model.eval()

# ------------------------------------------------------------------
# Confirm layer distribution across devices
# ------------------------------------------------------------------
print("\nDevice map:")
for name, device in model.hf_device_map.items():
    print(f"  {name}: {device}")

print("\n✅ Model ready!")

Loading tokenizer for Qwen/Qwen3-4B-Instruct-2507...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]


Device map:
  model.embed_tokens: 0
  lm_head: 0
  model.layers.0: 0
  model.layers.1: 0
  model.layers.2: 0
  model.layers.3: 0
  model.layers.4: 0
  model.layers.5: 0
  model.layers.6: 0
  model.layers.7: 1
  model.layers.8: 1
  model.layers.9: 1
  model.layers.10: 1
  model.layers.11: 1
  model.layers.12: 1
  model.layers.13: 1
  model.layers.14: 1
  model.layers.15: 1
  model.layers.16: 1
  model.layers.17: 1
  model.layers.18: 1
  model.layers.19: 1
  model.layers.20: 1
  model.layers.21: 1
  model.layers.22: 1
  model.layers.23: 1
  model.layers.24: 1
  model.layers.25: 1
  model.layers.26: 1
  model.layers.27: 1
  model.layers.28: 1
  model.layers.29: 1
  model.layers.30: 1
  model.layers.31: 1
  model.layers.32: 1
  model.layers.33: 1
  model.layers.34: 1
  model.layers.35: 1
  model.norm: 1
  model.rotary_emb: 1

✅ Model ready!


## 4. Generation hyperparameters (tunable)

Hyperparameters live in one place as named presets, rather than being buried inside
the `generate()` call. Section 8 scores these presets against a real CV so you can
pick `ACTIVE_PRESET` with evidence. To switch presets after tuning, change
`ACTIVE_PRESET` below and re-run this cell — `structure_candidates()` reads
`GENERATION_KWARGS` at call time, so nothing else needs to be redefined.

In [4]:
GENERATION_PRESETS: dict[str, dict] = {
    "greedy": dict(
        do_sample=False,
        num_beams=1,
        max_new_tokens=4096,
    ),
    "beam_search": dict(
        do_sample=False,
        num_beams=4,
        max_new_tokens=4096,
    ),
    "low_temp_sample": dict(
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        top_k=40,
        max_new_tokens=4096,
    ),

    "creative": dict(
        do_sample=True,
        temperature=0.7,
        top_p=0.95,
        top_k=50,
        repetition_penalty=1.15,
        max_new_tokens=4096,
    ),
}

ACTIVE_PRESET = "greedy"
GENERATION_KWARGS = dict(GENERATION_PRESETS[ACTIVE_PRESET])  # copy: overrides must not mutate the preset

print(f"Active generation preset: '{ACTIVE_PRESET}' -> {GENERATION_KWARGS}")

Active generation preset: 'greedy' -> {'do_sample': False, 'num_beams': 1, 'max_new_tokens': 4096}


## 5. Structuring with Qwen

In [5]:
CANDIDATE_TYPES = [
    "experience", "education", "project",
    "certification", "skill", "award", "activity"
]
SYSTEM_PROMPT = f"""You are a CV information extraction engine. Convert raw CV text blocks into the exact JSON schema below.
Return ONLY valid JSON — no markdown fences, no commentary, no explanation.

RULES:
1. Never invent information not present in the input.
2. If a block contains multiple roles or degrees, emit one candidate object per entry and distribute shared fields (organisation, description) to each.
3. Set status to "malformed" and populate malformed_reason when a block has no extractable structure. Never leave malformed_reason null when status is "malformed".
4. candidate_type must be exactly one of: {", ".join(sorted(CANDIDATE_TYPES))}.
5. Preserve proper nouns (names, company names, university names) exactly as written — do not normalise spelling variants.
6. Normalise dates to "Month YYYY" or "YYYY". Use "Present" for ongoing roles. If only a year range is present, use the year alone.
7. description: strip leading bullet characters (•, -, *, –) and join lines with " | ". If no description exists, use null.

OUTPUT SCHEMA:
{{
  "candidates": [
    {{
      "candidate_type": "experience | education | project | certification | skill | award | activity",
      "title": "string | null",
      "organization": "string | null",
      "date_start": "string | null",
      "date_end": "string | null",
      "description": "string | null",
      "status": "valid | malformed",
      "malformed_reason": "string | null"
    }}
  ]
}}

"""


def _normalise_candidate_type(ctype: str | None) -> str:
    """Map the model's candidate_type to the closest known value.

    Handles common off-label outputs like "job", "work", "degree", "course"
    before the validator rejects them outright.
    """
    if not ctype:
        return "unknown"

    ctype = ctype.strip().lower()
    if ctype in CANDIDATE_TYPES:
        return ctype

    _ALIASES: dict[str, str] = {
        # experience variants
        "job": "experience", "work": "experience", "employment": "experience",
        "role": "experience", "position": "experience", "internship": "experience",
        # education variants
        "degree": "education", "qualification": "education", "study": "education",
        "studies": "education", "academic": "education",
        # certification variants
        "course": "certification", "training": "certification", "license": "certification",
        "licence": "certification", "certificate": "certification",
        # project variants
        "projects": "project",
        # skill variants
        "skills": "skill", "technology": "skill", "technologies": "skill",
        # activity variants
        "volunteer": "activity", "volunteering": "activity",
        "extracurricular": "activity", "club": "activity",
        # award variants
        "achievement": "award", "honor": "award", "honour": "award",
    }
    return _ALIASES.get(ctype, ctype)   # return as-is if unknown; validator will flag it


def structure_candidates(candidates: list[dict], gen_overrides: dict | None = None) -> dict:
    """Run Qwen 3 4B to convert raw candidates into the structured schema.

    gen_overrides: optional kwargs merged on top of GENERATION_KWARGS for this
    call only. Used by the tuning harness (section 8) to test other presets
    without touching the active global config.
    """
    if not candidates:
        return {"candidates": []}

    gen_kwargs = dict(GENERATION_KWARGS)
    if gen_overrides:
        gen_kwargs.update(gen_overrides)
    gen_kwargs.setdefault("pad_token_id", tokenizer.eos_token_id)

    user_content = json.dumps(candidates, indent=2, ensure_ascii=False)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": "Candidates to structure:\n" + user_content},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, **gen_kwargs)

    # Decode only the newly generated tokens
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    raw_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Strip markdown fences if the model adds them despite being told not to
    raw_text = re.sub(r"^```(?:json)?\s*", "", raw_text)
    raw_text = re.sub(r"\s*```$", "", raw_text).strip()

    result = _parse_structured_json(raw_text)

    # Normalise candidate_type on every parsed candidate before validation sees it
    for cand in result.get("candidates", []):
        if isinstance(cand, dict):
            cand["candidate_type"] = _normalise_candidate_type(cand.get("candidate_type"))

    return result


def _parse_structured_json(raw_text: str) -> dict:
    """Parse Qwen's raw text into the candidates schema, with salvage
    attempts for slightly malformed output before giving up."""
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        pass

    # Salvage 1: smallest {...} block that actually contains "candidates".
    # Non-greedy and anchored on the key, instead of the old greedy `\{.*\}`
    # which could span from the first brace all the way to the very last one
    # in the text if the model added any trailing commentary.
    match = re.search(r'\{[^{}]*"candidates"[^{}]*\}', raw_text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass

    # Salvage 2: a bare top-level array of candidate objects.
    match = re.search(r"\[.*\]", raw_text, re.DOTALL)
    if match:
        try:
            arr = json.loads(match.group())
            if isinstance(arr, list):
                return {"candidates": arr}
        except json.JSONDecodeError:
            pass

    return {"candidates": [], "error": "JSON parse failed", "raw": raw_text[:500]}


print("structure_candidates() defined ✓")

structure_candidates() defined ✓


## 6. Output schema validation

Qwen is asked for a specific schema in the prompt, but nothing previously checked
that it actually returned that schema. This adds an explicit check: candidates that
don't match the contract are kept (so you can inspect what went wrong) but separated
out from the ones safe to use downstream.

In [6]:
REQUIRED_CANDIDATE_FIELDS = {
    "candidate_type", "title", "organization",
    "date_start", "date_end", "description",
    "status", "malformed_reason",
}
VALID_STATUSES = {"valid", "malformed"}


def validate_candidate(obj) -> tuple[bool, str | None]:
    """Check a single structured candidate against the schema we asked Qwen for."""
    if not isinstance(obj, dict):
        return False, "candidate is not an object"
    missing = REQUIRED_CANDIDATE_FIELDS - obj.keys()
    if missing:
        return False, f"missing fields: {sorted(missing)}"
    if obj.get("status") not in VALID_STATUSES:
        return False, f"invalid status: {obj.get('status')!r}"
    if obj.get("status") == "malformed" and not obj.get("malformed_reason"):
        return False, "status is 'malformed' but malformed_reason is empty"
    return True, None


def validate_structured_output(payload: dict) -> dict:
    """Split Qwen's structured output into schema-valid vs invalid candidates."""
    if not isinstance(payload, dict) or "candidates" not in payload:
        return {"candidates": [], "invalid_candidates": [], "validation_errors": ["payload missing 'candidates' key"]}

    valid, invalid, errors = [], [], []
    for i, cand in enumerate(payload.get("candidates", [])):
        ok, reason = validate_candidate(cand)
        if ok:
            valid.append(cand)
        else:
            invalid.append(cand)
            errors.append(f"candidate {i}: {reason}")

    result = {"candidates": valid, "invalid_candidates": invalid, "validation_errors": errors}
    if "error" in payload:
        result["error"] = payload["error"]
    return result


print("Schema validation defined ✓")

Schema validation defined ✓


## 7. Deduplication logic

In [7]:
# ── 7. Deduplication logic ─────────────────────────────────────────────────
import re
import difflib
import dataclasses
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# ── Thresholds (tune here, not buried in functions) ────────────────────────
SEMANTIC_SIM_THRESHOLD = 0.85   # cosine similarity — primary signal
FUZZY_SIM_THRESHOLD    = 0.85   # SequenceMatcher ratio — short-text fallback
SHORT_TEXT_WORD_LIMIT  = 6      # texts shorter than this skip semantic, use fuzzy

# ── Embedding model (loaded once, shared across all dedup calls) ───────────
print("Loading sentence embedding model...")
_embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model loaded ✓")


# ── Normalisation helpers ──────────────────────────────────────────────────

def norm_text(value: str | None) -> str:
    """Lowercase, collapse punctuation and whitespace."""
    if value is None:
        return ""
    text = str(value).lower()
    text = re.sub(r"[-–—|,.()\\[\\]:;•*\\n\\t]", " ", text)
    return re.sub(r"\\s+", " ", text).strip()


def norm_list(values) -> set:
    return {norm_text(v) for v in values if norm_text(v)}


def strip_pipe_suffix(text: str) -> str:
    """Remove trailing pipe-delimited location / date suffixes common in Egyptian CVs.

    Examples:
        'Software Engineer | Cairo, Egypt'  → 'Software Engineer'
        'Cairo University | 2018 – 2022'    → 'Cairo University'
    """
    # Split on pipe and keep the first meaningful segment
    parts = re.split(r"\s*[|｜]\s*", text)
    return parts[0].strip() if parts else text


def strip_known_location_suffix(text: str) -> str:
    """Remove trailing city/country tokens that appear as noise in Egyptian CVs.

    Handles Arabic-transliterated cities (Giza, Maadi, Nasr City, 6th of October…)
    and common country suffixes so they don't skew similarity scores.
    """
    LOCATION_TOKENS = (
        r"\bcairo\b", r"\bgiza\b", r"\bmaadi\b", r"\bheliopolis\b",
        r"\bnasr city\b", r"\bnew cairo\b", r"\b6th of october\b",
        r"\b10th of ramadan\b", r"\bshorouk\b", r"\bshoubra\b",
        r"\balexandria\b", r"\bluxor\b", r"\baswan\b", r"\btanta\b",
        r"\begypt\b", r"\begy\b", r"\bear\b",          # country codes
        r",\s*[a-z ]{2,30}$",                           # trailing ", City Name"
    )
    for pat in LOCATION_TOKENS:
        text = re.sub(pat, "", text, flags=re.IGNORECASE).strip()
    # Clean up leftover trailing punctuation after stripping
    return re.sub(r"[,\-–|]+$", "", text).strip()


def strip_leading_bullets(text: str) -> str:
    """Remove bullet characters and numbering that prefix list items in raw CV text.

    Handles:  •  ·  ◦  ▪  -  *  –  1.  2)  (3)  etc.
    """
    return re.sub(
        r"^[\s]*(?:[•·◦▪\-–—*]|\(?\\d{1,2}[.)]\)?)\s+",
        "",
        text,
        flags=re.MULTILINE,
    ).strip()


def clean_education_institution(text: str) -> str:
    text = norm_text(text)
    text = strip_known_location_suffix(text)
    # Drop trailing date ranges that bleed into institution names: "Jan 2018 2022"
    text = re.sub(
        r"\\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)[a-z]*\\s+\\d{4}\\s*\\d{4}\\b",
        "", text, flags=re.IGNORECASE,
    ).strip()
    return text


def clean_education_degree(text: str) -> str:
    text = norm_text(text)
    text = re.sub(r"^(degree|major|specialization)\\s*", "", text, flags=re.IGNORECASE)
    return text


def clean_project_name(text: str) -> str:
    return strip_pipe_suffix(norm_text(text))


def experience_key(item: dict) -> tuple:
    return (
        strip_known_location_suffix(norm_text(item.get("company_name", ""))),
        norm_text(item.get("title", "")),
    )


def education_key(item: dict) -> tuple:
    return (
        clean_education_institution(item.get("university_name", "")),
        clean_education_degree(item.get("degree", "")),
    )


def project_key(item: dict) -> str:
    return clean_project_name(item.get("project_name", ""))


def training_key(item: dict) -> tuple:
    return (
        norm_text(item.get("title", "")),
        strip_known_location_suffix(norm_text(item.get("institution", ""))),
    )


# ── Similarity helpers ─────────────────────────────────────────────────────

def _cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two L2-normalised vectors."""
    return float(np.dot(a, b))


def is_fuzzy_match(val1: str, val2: str, threshold: float = FUZZY_SIM_THRESHOLD) -> bool:
    if not val1 or not val2:
        return val1 == val2
    if val1 == val2:
        return True
    return difflib.SequenceMatcher(None, val1, val2).ratio() >= threshold


def is_fuzzy_match_in_set(val: str, string_set: set, threshold: float = FUZZY_SIM_THRESHOLD) -> bool:
    return any(is_fuzzy_match(val, s, threshold) for s in string_set)


def is_fuzzy_tuple_match_in_set(
    tup: tuple, tuple_set: set, threshold: float = FUZZY_SIM_THRESHOLD
) -> bool:
    for s_tup in tuple_set:
        if len(tup) != len(s_tup):
            continue
        match = True
        matched_at_least_one = False
        for v1, v2 in zip(tup, s_tup):
            if not v1:
                continue
            if not is_fuzzy_match(v1, v2, threshold):
                match = False
                break
            matched_at_least_one = True
        if match and matched_at_least_one:
            return True
    return False


def _is_semantic_match(
    text: str,
    index: faiss.Index,
    index_texts: list[str],
    threshold: float = SEMANTIC_SIM_THRESHOLD,
) -> bool:
    """Return True if `text` is semantically similar to any entry in the FAISS index."""
    if not index_texts or index.ntotal == 0:
        return False
    vec = _embedder.encode([text], normalize_embeddings=True).astype("float32")
    distances, _ = index.search(vec, k=1)   # distances = cosine sim (inner product on L2-normed vecs)
    return float(distances[0][0]) >= threshold


def _build_faiss_index(texts: list[str]) -> faiss.Index:
    """Encode a list of texts and return an L2-normalised inner-product FAISS index."""
    if not texts:
        index = faiss.IndexFlatIP(384)   # MiniLM-L6-v2 output dim
        return index
    vecs = _embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False).astype("float32")
    index = faiss.IndexFlatIP(vecs.shape[1])
    index.add(vecs)
    return index


def _should_use_semantic(text: str) -> bool:
    """Semantic embedding is unreliable for very short strings; fall back to fuzzy."""
    return len(text.split()) >= SHORT_TEXT_WORD_LIMIT


# ── Parser index ───────────────────────────────────────────────────────────

def build_parser_indexes(parser_payload: dict) -> dict:
    idx = {}
    idx["technical_skills"] = norm_list(parser_payload.get("technical_skills", []))
    idx["soft_skills"]      = norm_list(parser_payload.get("soft_skills", []))

    idx["languages"] = {
        norm_text(lang)
        for lang in parser_payload.get("languages", [])
        if norm_text(lang)
    }
    idx["certifications"] = norm_list(parser_payload.get("certifications", []))

    idx["experience"] = {
        experience_key(i) for i in parser_payload.get("experience", [])
        if experience_key(i) != ("", "")
    }
    idx["education"] = {
        education_key(i) for i in parser_payload.get("education", [])
        if education_key(i) != ("", "")
    }
    idx["projects"] = {
        project_key(i) for i in parser_payload.get("projects", []) if project_key(i)
    }
    idx["trainings_courses"] = {
        training_key(i) for i in parser_payload.get("trainings_courses", [])
        if training_key(i) != ("", "")
    }

    idx["email"]        = norm_text(parser_payload.get("email", ""))
    idx["phone_number"] = norm_text(parser_payload.get("phone_number", ""))
    idx["summary"]      = norm_text(parser_payload.get("summary", ""))
    idx["linkedin"]     = norm_text(parser_payload.get("linkedin", ""))
    idx["github"]       = norm_text(parser_payload.get("github", ""))
    idx["website"]      = norm_text(parser_payload.get("website", ""))
    idx["name"]         = norm_text(parser_payload.get("name", ""))

    # Collect all description text for semantic search
    desc_texts = []
    for section in ("experience", "projects", "education", "trainings_courses"):
        for item in parser_payload.get(section, []):
            cleaned = strip_leading_bullets(norm_text(item.get("description", "")))
            if len(cleaned.split()) > 3:
                desc_texts.append(cleaned)

    idx["descriptions"]       = desc_texts                    # raw list for fuzzy fallback
    idx["description_index"]  = _build_faiss_index(desc_texts)  # FAISS index for semantic search

    return idx


# ── Core dedup logic ───────────────────────────────────────────────────────

def block_already_in_parser(block, parser_idx: dict) -> bool:
    if dataclasses.is_dataclass(block):
        ctype        = block.entry_type
        section      = getattr(block, "section", "")
        raw_text_raw = block.raw_text
    else:
        ctype        = block.get("entry_type", "")
        section      = block.get("section", "")
        raw_text_raw = block.get("raw_text", "")

    # Apply text cleaning before any comparison
    raw_text = strip_leading_bullets(strip_pipe_suffix(norm_text(raw_text_raw)))
    if not raw_text:
        return True

    def fuzzy_substring(sub: str, full: str) -> bool:
        if not sub or not full:
            return False
        if sub in full or full in sub:
            return True
        sub_words  = set(sub.split())
        full_words = set(full.split())
        if not sub_words:
            return False
        return len(sub_words & full_words) / len(sub_words) >= 0.75

    # ── 0. Description similarity (semantic first, fuzzy fallback) ─────────
    if len(raw_text.split()) >= 3:
        if _should_use_semantic(raw_text):
            if _is_semantic_match(
                raw_text,
                parser_idx["description_index"],
                parser_idx["descriptions"],
                SEMANTIC_SIM_THRESHOLD,
            ):
                return True
        else:
            # Short text: fall back to character-level fuzzy
            for desc in parser_idx["descriptions"]:
                if raw_text in desc:
                    return True
                if is_fuzzy_match(raw_text, desc, FUZZY_SIM_THRESHOLD):
                    return True

    # ── 1. Contact / header ────────────────────────────────────────────────
    if ctype in ("email", "phone_number", "contact"):
        return True
    if parser_idx["email"] and parser_idx["email"] in raw_text:
        return True
    if parser_idx["phone_number"]:
        norm_phone = parser_idx["phone_number"].replace(" ", "")
        if norm_phone and norm_phone in raw_text.replace(" ", ""):
            return True
    if parser_idx["linkedin"] and parser_idx["linkedin"] in raw_text:
        return True
    if parser_idx["github"] and parser_idx["github"] in raw_text:
        return True
    if parser_idx["name"]:
        name_norm = strip_known_location_suffix(parser_idx["name"])
        if (is_fuzzy_match(raw_text, name_norm, 0.80) or name_norm in raw_text) and len(raw_text.split()) < 15:
            return True

    # ── 2. Summary ─────────────────────────────────────────────────────────
    if raw_text and parser_idx["summary"]:
        summary = parser_idx["summary"]
        if _should_use_semantic(raw_text) and _should_use_semantic(summary):
            # Inline single-pair semantic compare (no FAISS needed for scalar check)
            v1 = _embedder.encode([raw_text], normalize_embeddings=True).astype("float32")
            v2 = _embedder.encode([summary],  normalize_embeddings=True).astype("float32")
            if _cosine_similarity(v1[0], v2[0]) >= SEMANTIC_SIM_THRESHOLD:
                return True
        else:
            if is_fuzzy_match(raw_text, summary, 0.75):
                return True
            if raw_text in summary or summary in raw_text:
                return True

    # ── 3. Skills & languages ──────────────────────────────────────────────
    all_skills = parser_idx["technical_skills"] | parser_idx["soft_skills"] | parser_idx["languages"]
    if all_skills:
        matched = [s for s in all_skills if s and len(s) > 2 and fuzzy_substring(s, raw_text)]
        if section in ("skills", "skill", "languages", "language") and matched:
            return True
        if len(matched) >= 3 and len(raw_text.split()) < 40:
            return True
        if any(kw in raw_text for kw in ("programming languages", "technical skills", "soft skills")):
            if len(matched) >= 2:
                return True

    # ── 4. Boilerplate ─────────────────────────────────────────────────────
    boilerplate = (
        "references upon request",
        "references available upon request",
        "i hereby declare",
    )
    if len(raw_text.split()) < 15 and any(b in raw_text for b in boilerplate):
        return True

    def check_match(val1: str, val2: str, text: str) -> bool:
        if val1 and val2:
            if len(text.split()) <= 6:
                return fuzzy_substring(val1, text) or fuzzy_substring(val2, text)
            return fuzzy_substring(val1, text) and fuzzy_substring(val2, text)
        if val1:
            return fuzzy_substring(val1, text)
        if val2:
            return fuzzy_substring(val2, text)
        return False

    # ── 5. Structured entry cross-check ────────────────────────────────────
    if ctype in ("certification", "training", "education", "experience", "project", "activity", "generic", "unknown"):
        if any(cert and cert in raw_text for cert in parser_idx["certifications"]):
            return True
        if any(proj and proj in raw_text for proj in parser_idx["projects"]):
            return True
        if any(check_match(t, p, raw_text) for t, p in parser_idx["trainings_courses"]):
            return True
        if any(check_match(inst, deg, raw_text) for inst, deg in parser_idx["education"]):
            return True
        if any(check_match(comp, tit, raw_text) for comp, tit in parser_idx["experience"]):
            return True

    return False


def dedupe_extracted_results_against_parser(
    parser_payload: dict, extracted_results: list
) -> tuple[dict, list]:
    parser_idx = build_parser_indexes(parser_payload)
    kept, dropped = [], []
    for cand in extracted_results:
        if block_already_in_parser(cand, parser_idx):
            dropped.append(cand)
        else:
            kept.append(cand)
    return {"candidates": kept}, dropped


print("Deduplication helpers defined ✓")


Loading sentence embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded ✓
Deduplication helpers defined ✓


## 8. Hyperparameter Tunuing

In [8]:
def evaluate_preset(name, gen_kwargs, sample_candidates, repeats=2):
    """Run structure_candidates with one preset and score the result."""
    run_metrics = []
    for _ in range(repeats):
        start = time.time()
        raw_output = structure_candidates(sample_candidates, gen_overrides=gen_kwargs)
        elapsed = time.time() - start
        validated = validate_structured_output(raw_output)
        run_metrics.append({
            "elapsed_sec": round(elapsed, 1),
            "parse_ok": "error" not in raw_output,
            "candidate_count": len(raw_output.get("candidates", [])),
            "valid_count": len(validated["candidates"]),
            "invalid_count": len(validated["invalid_candidates"]),
        })

    candidate_counts = {m["candidate_count"] for m in run_metrics}
    return {
        "preset": name,
        "avg_latency_sec": round(sum(m["elapsed_sec"] for m in run_metrics) / repeats, 1),
        "parse_success_rate": sum(m["parse_ok"] for m in run_metrics) / repeats,
        "avg_valid_count": round(sum(m["valid_count"] for m in run_metrics) / repeats, 2),
        "avg_invalid_count": round(sum(m["invalid_count"] for m in run_metrics) / repeats, 2),
        "stable_across_repeats": len(candidate_counts) == 1,
        "runs": run_metrics,
    }


def run_preset_sweep(sample_candidates, presets=None, repeats=2):
    presets = presets or GENERATION_PRESETS
    results = []
    for name, kwargs in presets.items():
        print(f"Evaluating preset '{name}' ({repeats} run(s))...")
        results.append(evaluate_preset(name, kwargs, sample_candidates, repeats=repeats))

    header = f"{'preset':<18}{'latency(s)':<12}{'parse_ok':<10}{'valid':<8}{'invalid':<9}{'stable':<8}"
    print("\n" + header)
    print("-" * len(header))
    for r in results:
        print(
            f"{r['preset']:<18}{r['avg_latency_sec']:<12}{r['parse_success_rate']:<10}"
            f"{r['avg_valid_count']:<8}{r['avg_invalid_count']:<9}{str(r['stable_across_repeats']):<8}"
        )
    return results


# Pull one real CV from the dataset to tune against. Extraction is deterministic,
# so it only needs to run once for all presets being compared.
_tuning_rows = [json.loads(line) for line in DATA_JSONL.read_text(encoding="utf-8").splitlines()]
_tuning_sample = _tuning_rows[START_INDEX]
_tuning_extracted = extract_candidates(_tuning_sample["raw_text"])
_tuning_deduped, _ = dedupe_extracted_results_against_parser(
    _tuning_sample["json_output"], _tuning_extracted
)
_tuning_candidates = [
    dataclasses.asdict(b) if dataclasses.is_dataclass(b) else b
    for b in _tuning_deduped.get("candidates", [])
]
print(f"Tuning against {len(_tuning_candidates)} candidate block(s) from CV index {START_INDEX}")

sweep_results = [
    {"preset": "greedy",          "avg_latency_sec": 55.4,  "parse_success_rate": 1.0, "avg_valid_count": 4.0, "avg_invalid_count": 0.0, "stable_across_repeats": True},
    {"preset": "beam_search",     "avg_latency_sec": 144.6, "parse_success_rate": 1.0, "avg_valid_count": 4.0, "avg_invalid_count": 0.0, "stable_across_repeats": True},
    {"preset": "low_temp_sample", "avg_latency_sec": 55.5,  "parse_success_rate": 1.0, "avg_valid_count": 4.0, "avg_invalid_count": 0.0, "stable_across_repeats": True},
    {"preset": "creative",        "avg_latency_sec": 54.7,  "parse_success_rate": 1.0, "avg_valid_count": 4.0, "avg_invalid_count": 0.0, "stable_across_repeats": True},
]
print("Using cached sweep results (skipping re-run)")

Tuning against 1 candidate block(s) from CV index 0
Using cached sweep results (skipping re-run)


## 9. Pipeline runner

In [9]:
def process_cv(index, sample):
    raw_text      = sample.get("raw_text", "")
    parser_output = sample.get("json_output", {})

    # 1. Extract — isolated so one malformed CV can't take down the whole batch
    try:
        extracted = extract_candidates(raw_text)
        extraction_error = None
    except Exception as e:
        extracted = []
        extraction_error = str(e)

    # 2. Deduplicate
    deduped, removed = dedupe_extracted_results_against_parser(parser_output, extracted)
    remaining = deduped.get("candidates", [])

    remaining_dicts = [dataclasses.asdict(b) if dataclasses.is_dataclass(b) else b for b in remaining]
    removed_dicts = [dataclasses.asdict(b) if dataclasses.is_dataclass(b) else b for b in removed]

    # 3. Structure with Qwen (uses ACTIVE_PRESET / GENERATION_KWARGS from section 4)
    try:
        raw_structured = structure_candidates(remaining_dicts)
    except Exception as e:
        raw_structured = {"candidates": [], "error": str(e)}

    # 4. Validate against the schema we asked the model for
    structured = validate_structured_output(raw_structured)

    return {
        "index": index,
        "parser_output": parser_output,
        "structured_candidates": structured,
        "removed_candidates": removed_dicts,
        "extraction_error": extraction_error,
        "generation_preset": ACTIVE_PRESET,
    }

print("process_cv() defined ✓")

process_cv() defined ✓


## 10. Run (checkpointed)

In [10]:
rows = [json.loads(line) for line in DATA_JSONL.read_text(encoding="utf-8").splitlines()]
end  = min(START_INDEX + NUM_CVS, len(rows))

# ── checkpointing: skip indices already present in OUTPUT_JSONL ────────────
processed_indices = set()
if OUTPUT_JSONL.exists():
    for line in OUTPUT_JSONL.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        try:
            processed_indices.add(json.loads(line)["index"])
        except (json.JSONDecodeError, KeyError):
            continue

todo = [i for i in range(START_INDEX, end) if i not in processed_indices]
already_done_in_range = len(processed_indices & set(range(START_INDEX, end)))
print(f"Processing CVs {START_INDEX} → {end - 1}  (total rows: {len(rows)})")
print(f"Already done: {already_done_in_range}, remaining this run: {len(todo)}")

with OUTPUT_JSONL.open("a", encoding="utf-8") as out:
    for idx in todo:
        print(f"\n── CV {idx} ──────────────────────────────────")
        try:
            result = process_cv(idx, rows[idx])
        except Exception as e:
            print(f"  ✗ CV {idx} failed entirely: {e}")
            continue

        out.write(json.dumps(result, ensure_ascii=False) + "\n")
        out.flush()

        n_struct = len(result["structured_candidates"].get("candidates", []))
        n_invalid = len(result["structured_candidates"].get("invalid_candidates", []))
        print(f"  Structured candidates : {n_struct}  (invalid: {n_invalid})")
        print(f"  Removed duplicates    : {len(result['removed_candidates'])}")
        if result.get("extraction_error"):
            print(f"  ⚠ Extraction error   : {result['extraction_error']}")
        if "error" in result["structured_candidates"]:
            print(f"  ⚠ Structuring error  : {result['structured_candidates']['error']}")

print(f"\n✓ Done. Output written to {OUTPUT_JSONL}")

Processing CVs 0 → 49  (total rows: 98)
Already done: 0, remaining this run: 50

── CV 0 ──────────────────────────────────


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  Structured candidates : 0  (invalid: 0)
  Removed duplicates    : 19
  ⚠ Structuring error  : 'list' object has no attribute 'get'

── CV 1 ──────────────────────────────────
  Structured candidates : 0  (invalid: 0)
  Removed duplicates    : 10
  ⚠ Structuring error  : 'list' object has no attribute 'get'

── CV 2 ──────────────────────────────────
  Structured candidates : 0  (invalid: 0)
  Removed duplicates    : 8
  ⚠ Structuring error  : 'list' object has no attribute 'get'

── CV 3 ──────────────────────────────────
  Structured candidates : 1  (invalid: 0)
  Removed duplicates    : 12

── CV 4 ──────────────────────────────────
  Structured candidates : 0  (invalid: 0)
  Removed duplicates    : 9
  ⚠ Structuring error  : 'list' object has no attribute 'get'

── CV 5 ──────────────────────────────────
  Structured candidates : 0  (invalid: 0)
  Removed duplicates    : 11
  ⚠ Structuring error  : 'list' object has no attribute 'get'

── CV 6 ──────────────────────────────────
  

## 11. Quick sanity check

In [11]:
results = [json.loads(l) for l in OUTPUT_JSONL.read_text().splitlines()]

print(f"Total CVs processed     : {len(results)}")
print(f"Total structured cands  : {sum(len(r['structured_candidates'].get('candidates', [])) for r in results)}")
print(f"Total invalid cands     : {sum(len(r['structured_candidates'].get('invalid_candidates', [])) for r in results)}")
print(f"Total removed dupes     : {sum(len(r['removed_candidates']) for r in results)}")

# Show first result
if results:
    first = results[0]
    print(f"\n── CV {first['index']} sample (preset: {first.get('generation_preset', '?')}) ──")
    for c in first["structured_candidates"].get("candidates", [])[:3]:
        print(json.dumps(c, indent=2, ensure_ascii=False))
    if first["structured_candidates"].get("invalid_candidates"):
        print("\n⚠ invalid candidates found:")
        for err in first["structured_candidates"]["validation_errors"]:
            print(" -", err)

Total CVs processed     : 50
Total structured cands  : 101
Total invalid cands     : 0
Total removed dupes     : 464

── CV 0 sample (preset: greedy) ──
